# 🫁 Respiratory Sound Analysis

> Analysis of respiratory sounds using advanced signal processing techniques

---

## 📦 Installation & Setup

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas

In [ ]:
import sys
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import wavfile

from google.colab import drive

MAX_FILES: int | None = None
EMBEDDING_DIM: int = 3
TIME_DELAY: int = 1


In [ ]:
drive.mount('/content/drive')


## ⚙️ Configure Kaggle & Download Dataset

In [ ]:
KAGGLE_DIR = Path.home() / '.kaggle'
KAGGLE_JSON = KAGGLE_DIR / 'kaggle.json'
KAGGLE_DRIVE = Path('/content/drive/MyDrive/kaggle.json')

KAGGLE_DIR.mkdir(exist_ok=True)
shutil.copy(KAGGLE_DRIVE, KAGGLE_JSON)
KAGGLE_JSON.chmod(0o600)


In [ ]:
DATASET_PATH = Path('/content/respiratory_sound_dataset')
DATASET_DRIVE = Path('/content/drive/MyDrive/respiratory_sound_dataset')

!cp -r {DATASET_DRIVE} /content/ 2>/dev/null && echo "✅ Loaded from Drive cache" || \
    (kaggle datasets download -d vbookshelf/respiratory-sound-database -q && \
     unzip -q respiratory-sound-database.zip -d {DATASET_PATH} && \
     rm respiratory-sound-database.zip && \
     cp -r {DATASET_PATH} /content/drive/MyDrive/ && \
     echo "✅ Downloaded and cached to Drive")


## 📥 Clone Analysis Repository

In [ ]:
REPO_PATH = Path('/content/course_paper')

!git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
!git -C {REPO_PATH} sparse-checkout set app

sys.path.insert(0, str(REPO_PATH))

from app.data_preprocessing import extract_features_batch


## 🔊 Extract Features and Create DataFrame

In [ ]:
audio_dir = next(
    matches[0] for pattern in [
        '**/audio_and_txt_files', 
        '**/Respiratory_Sound_Database/**/audio_and_txt_files'
    ]
    if (matches := list(DATASET_PATH.glob(pattern)))
)

all_wav_files = sorted(audio_dir.glob("*.wav"))
wav_files = all_wav_files if MAX_FILES is None else all_wav_files[:MAX_FILES]

signals = {}

for wav_file in wav_files:
    sample_rate, signal_data = wavfile.read(str(wav_file))
    signal_data = signal_data.astype(np.float64)
    signal_data = signal_data[:, 0] if signal_data.ndim > 1 else signal_data
    
    signals[wav_file.name] = {
        'signal': signal_data,
        'sample_rate': sample_rate
    }

print(f"✅ Loaded {len(signals)} files @ {sample_rate} Hz")


In [ ]:
diagnosis_files = list(DATASET_PATH.rglob('patient_diagnosis.csv'))
diagnosis_file = diagnosis_files[0]
print(f"📋 Found: {diagnosis_file}")

diagnosis_df = pd.read_csv(diagnosis_file)
patient_col, diagnosis_col = diagnosis_df.columns[0], diagnosis_df.columns[1]
diagnosis_map = dict(zip(diagnosis_df[patient_col], diagnosis_df[diagnosis_col]))

rows = extract_features_batch(signals, embedding_dim=EMBEDDING_DIM, time_delay=TIME_DELAY)

for row in rows:
    row['diagnosis'] = diagnosis_map.get(row['patient_id'], 'Unknown')

results_df = pd.DataFrame(rows)

print(f"\n✅ Extracted features: {results_df.shape}")
print(f"\n📋 Diagnoses:\n{results_df['diagnosis'].value_counts()}")
display(results_df.head())


In [ ]:
feature_cols = [
    'low_freq_energy', 'mid_freq_energy', 'high_freq_energy',
    'whistle_strength', 'spectral_centroid', 'peak_frequency',
    'entropy', 'complexity'
]

avg_by_diagnosis = results_df.groupby('diagnosis')[feature_cols].mean().reset_index()
display(avg_by_diagnosis)


## 📊 Visualizations

In [ ]:
DIAGNOSIS_COLORS = [
    '#FF4757', '#1DD1A1', '#5F9EFF', '#FFA502', '#00D2C3', 
    '#FFE66D', '#C56CF0', '#54A0FF', '#FF6B9D', '#48DBFB', 
    '#F8EFBA', '#1E90FF', '#FF7979'
]
PLOT_STYLE = {
    'grid_alpha': 0.15,
    'bar_alpha': 0.85,
    'line_width': 2.0,
    'title_fontsize': 15,
    'label_fontsize': 12,
    'value_fontsize': 9,
    'marker_size': 4,
    'marker_density': 20,
    'ylim_multiplier': 1.20,
    'scatter_size': 300
}

plt.style.use('dark_background')

plt.rcParams['figure.facecolor'] = '#0B1929'
plt.rcParams['axes.facecolor'] = '#1A2332'
plt.rcParams['savefig.facecolor'] = '#0B1929'


In [ ]:
def save_plot(filename: str, dpi: int = 600):
    from google.colab import files
    local_path = f'/content/{filename}'
    plt.savefig(local_path, dpi=dpi, bbox_inches='tight')
    files.download(local_path)
    print(f'⬇️ Downloaded: {filename}')

In [ ]:
num_diagnoses = len(avg_by_diagnosis)
fig_width = max(16, num_diagnoses * 1.5)
fig, axes = plt.subplots(2, 4, figsize=(fig_width, 8))
axes = axes.flatten()

for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    
    diagnoses = avg_by_diagnosis['diagnosis']
    values = avg_by_diagnosis[feature]
    
    colors = [DIAGNOSIS_COLORS[i % len(DIAGNOSIS_COLORS)] for i in range(num_diagnoses)]
    bars = ax.bar(range(num_diagnoses), values, color=colors, alpha=PLOT_STYLE['bar_alpha'], 
                  width=0.7 if num_diagnoses <= 8 else 0.6)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', 
                fontsize=PLOT_STYLE['value_fontsize'] if num_diagnoses <= 8 else 8,
                fontweight='600')
    
    ax.set_title(feature.replace('_', ' ').title(), fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks(range(num_diagnoses))
    ax.set_xticklabels(diagnoses, rotation=45, ha='right', 
                       fontsize=PLOT_STYLE['value_fontsize'] if num_diagnoses <= 8 else 7)
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], axis='y', zorder=0)
    ax.set_ylim(0, values.max() * PLOT_STYLE['ylim_multiplier'])
    ax.set_axisbelow(True)

plt.suptitle('Feature Comparison Across Diagnoses', fontsize=PLOT_STYLE['title_fontsize']+2, 
             fontweight='bold', y=0.995)
plt.tight_layout()
save_plot('feature_comparison.png', dpi=600)
plt.show()


In [ ]:
min_length = min(len(data['signal']) for data in signals.values())
sample_rate = list(signals.values())[0]['sample_rate']

signals_by_diagnosis = defaultdict(list)
for filename, data in signals.items():
    patient_id = int(filename.split('_')[0])
    diagnosis = diagnosis_map.get(patient_id, 'Unknown')
    signal = data['signal'][:min_length]
    signals_by_diagnosis[diagnosis].append(signal)

for idx, diagnosis in enumerate(avg_by_diagnosis['diagnosis']):
    signals_array = np.array(signals_by_diagnosis[diagnosis])
    mean_signal = np.mean(signals_array, axis=0)
    std_signal = np.std(signals_array, axis=0)
    time_seconds = np.arange(len(mean_signal)) / sample_rate
    file_count = len(signals_by_diagnosis[diagnosis])
    color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]
    
    fig, ax = plt.subplots(figsize=(12, 4.5))
    
    ax.fill_between(time_seconds, mean_signal - std_signal, mean_signal + std_signal, 
                    color=color, alpha=0.3, label='±1 SD', zorder=2)
    
    ax.plot(time_seconds, mean_signal, color=color, linewidth=PLOT_STYLE['line_width']+0.5, 
            label='Mean', alpha=0.95, zorder=3)
    
    marker_interval = max(1, len(time_seconds) // PLOT_STYLE['marker_density'])
    ax.plot(time_seconds[::marker_interval], mean_signal[::marker_interval], 
           'o', color=color, markersize=PLOT_STYLE['marker_size'], zorder=4)
    
    duration = len(mean_signal) / sample_rate
    ax.set_title(f'{diagnosis} - (observations = {file_count}, duration = {duration:.2f}s)', 
                fontsize=PLOT_STYLE['title_fontsize'], fontweight='bold', pad=15)
    ax.set_xlabel('Time (seconds)', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
    ax.set_ylabel('Amplitude', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], zorder=0)
    ax.set_axisbelow(True)
    
    legend = ax.legend(loc='upper right', fontsize=10, framealpha=0.9, fancybox=True, shadow=False)
    legend.get_frame().set_linewidth(1.0)
    
    plt.tight_layout()
    save_plot(f'{diagnosis}_signal.png', dpi=600)
    plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

for idx, row in avg_by_diagnosis.iterrows():
    diagnosis = row['diagnosis']
    entropy = row['entropy']
    complexity = row['complexity']
    color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]
    
    ax.scatter(entropy, complexity, s=PLOT_STYLE['scatter_size'], color=color, 
              alpha=PLOT_STYLE['bar_alpha'], label=diagnosis, zorder=3)
    
    ax.annotate(diagnosis, (entropy, complexity), 
               xytext=(10, 10), textcoords='offset points',
               fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3),
               arrowprops=dict(arrowstyle='-', lw=0.8, alpha=0.6))

ax.set_xlabel('Entropy', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Complexity', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Entropy X Complexity', fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], linestyle='--', zorder=0)
ax.set_axisbelow(True)

legend = ax.legend(loc='best', fontsize=10, framealpha=0.95, fancybox=True, shadow=False)
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
save_plot('entropy_complexity.png', dpi=600)
plt.show()
